# 12.6 Other display features for models and instances

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

We gather in this section various utility commands and other features for displaying
information about models or about problem instances generated from models.

Two commands `let` you review an AMPL `model` from the command-line: show lists
the names of `model` components and displays the definitions of individual components,
while `xref` lists all components that depend on a given component. The expand command
displays selected objectives and constraints that AMPL has generated from a `model`
and `data`, or analogous information for variables. AMPL's "generic" names for variables,
constraints, or objectives permit listings or tests that apply to all variables, constraints, or
objectives.
#### Displaying model components: the show command:

By itself, the show command lists the names of all components of the current `model`:

```ampl
ampl: model multmip3.mod;
ampl: show;
parameters: demand fcost limit maxserve minload supply vcost
sets:   DEST   ORIG   PROD
variables:   Trans   Use
constraints:   Demand    Max_Serve Min_Ship   Multi   Supply
objective:   Total_Cost
checks: one, called check 1.
```

This `display` may be restricted to components of one or more types:

```ampl
ampl: show vars;
variables:   Trans   Use
ampl: show obj, constr;
objective:   Total_Cost
constraints:   Demand    Max_Serve   Min_Ship   Multi   Supply
```

The show command can also `display` the declarations of individual components, saving
you the trouble of looking them up in the `model` file:

```ampl
ampl: show Total_Cost;
minimize Total_Cost: sum{i in ORIG, j in DEST, p in PROD}
vcost[i,j,p]*Trans[i,j,p] + sum{i in ORIG, j in DEST}
fcost[i,j]*Use[i,j];
ampl: show vcost, fcost, Trans;
param vcost{ORIG, DEST, PROD} >= 0;
param fcost{ORIG, DEST} >= 0;
var Trans{ORIG, DEST, PROD} >= 0;
```

If an item following show is the name of a component in the current `model`, the declaration
of that component is displayed. Otherwise, the item is interpreted as a component
type according to its first letter or two; see Section A.19.1. (Displayed declarations may
differ in inessential ways from their appearance in your `model` file, due to transformations
that AMPL performs when the `model` is parsed and translated.)
Since the `check` statements in a `model` do not have names, AMPL numbers them in
the order that they appear. Thus to see the third `check` statement you would type

```ampl
ampl: show check 3;
check{p in PROD} :
       sum{i in ORIG} supply[i,p] == sum{j in DEST} demand[j,p];
```

By itself, show checks indicates the number of `check` statements in the `model`.
Displaying `model` dependencies: the `xref` command
The `xref` command lists all `model` components that depend on a specified component,
either directly (by referring to it) or indirectly (by referring to its dependents). If
more than one component is given, the dependents are listed separately for each. Here is
an example from multmip3.mod:

```ampl
ampl: xref demand, Trans;
# 2 entities depend on demand:
check 1
Demand
# 5 entities depend on Trans:
Total_Cost
Supply
Demand
Multi
Min_Ship
```

In general, the command is simply the keyword `xref` followed by a comma-separated
list of any combination of set, parameter, variable, `objective` and constraint names.

Displaying `model` instances: the expand command
In checking a `model` and its `data` for correctness, you may want to look at some of the
specific constraints that AMPL is generating. The expand command displays all constraints
in a given indexed collection or specific constraints that you identify:

```ampl
ampl: model nltrans.mod;
ampl: data nltrans.dat;
ampl: expand Supply;
subject to Supply['GARY']:
              Trans['GARY','FRA'] + Trans['GARY','DET'] +
              Trans['GARY','LAN'] + Trans['GARY','WIN'] +
              Trans['GARY','STL'] + Trans['GARY','FRE'] +
              Trans['GARY','LAF'] = 1400;
subject to Supply['CLEV']:
              Trans['CLEV','FRA'] + Trans['CLEV','DET'] +
              Trans['CLEV','LAN'] + Trans['CLEV','WIN'] +
              Trans['CLEV','STL'] + Trans['CLEV','FRE'] +
              Trans['CLEV','LAF'] = 2600;
subject to Supply['PITT']:
              Trans['PITT','FRA'] + Trans['PITT','DET'] +
              Trans['PITT','LAN'] + Trans['PITT','WIN'] +
              Trans['PITT','STL'] + Trans['PITT','FRE'] +
              Trans['PITT','LAF'] = 2900;
```

(See Figures 18-4 and 18-5.) The ordering of terms in an expanded constraint does not
necessarily correspond to the order of the symbolic terms in the constraint's declaration.

Objectives may be expanded in the same way:

```ampl
ampl: expand Total_Cost;
minimize Total_Cost:
       39*Trans['GARY','FRA']/(1 - Trans['GARY','FRA']/500) + 14*
       Trans['GARY','DET']/(1 - Trans['GARY','DET']/1000) + 11*
       Trans['GARY','LAN']/(1 - Trans['GARY','LAN']/1000) + 14*
       Trans['GARY','WIN']/(1 - Trans['GARY','WIN']/1000) + 16*
       ... 15 lines omitted
       Trans['PITT','FRE']/(1 - Trans['PITT','FRE']/500) + 20*
       Trans['PITT','LAF']/(1 - Trans['PITT','LAF']/900);
```

When expand is applied to a variable, it lists all of the nonzero coefficients of that
variable in the linear terms of objectives and constraints:

```ampl
ampl: expand Trans;
Coefficients of Trans['GARY','FRA']:
              Supply['GARY'] 1
              Demand['FRA']   1
              Total_Cost      0 + nonlinear

Coefficients of Trans['GARY','DET']:
              Supply['GARY'] 1
              Demand['DET']   1
              Total_Cost      0 + nonlinear

Coefficients of Trans['GARY','LAN']:
              Supply['GARY'] 1
              Demand['LAN']   1
              Total_Cost      0 + nonlinear

Coefficients of Trans['GARY','WIN']:
              Supply['GARY'] 1
              Demand['WIN']  1
              Total_Cost     0 + nonlinear
... 17 terms omitted
```

When a variable also appears in nonlinear expressions within an `objective` or a constraint,
the term + nonlinear is appended to represent those expressions.

The command expand alone produces an expansion of all variables, objectives and
constraints in a `model`. Since a single expand command can produce a very long listing,
you may want to redirect its output to a file by placing >filename at the end as explained
in [Section 12.7](./12_7_general_output_commands.ipynb) below.

The formatting of numbers in the expanded output is governed by the options
expand_precision and expand_round, which work like the `display`
command's display_precision and display_round described in [Section 12.3](./12_3_numeric_options_for_display.ipynb).

The output of expand reflects the "modeler's view" of the problem; it is based on
the `model` and `data` as they were initially read and translated. But AMPL's presolve phase
(Section 14.1) may make significant simplifications to the problem before it is sent to the
solver. To see the expansion of the "solver's view" of the problem following presolve,
use the keyword solexpand in place of expand.
**Generic synonyms for variables, constraints and objectives**

Occasionally it is useful to make a listing or a test that applies to all variables, constraints,
or objectives. For this purpose, AMPL provides automatically updated parameters
that hold the numbers of variables, constraints, and objectives in the currently generated
problem instance:

```ampl
_nvars       number of variables in the current problem
_ncons       number of constraints in the current problem
_nobjs       number of objectives in the current problem
```

Correspondingly indexed parameters contain the AMPL names of all the components:

```ampl
_varname{1.._nvars}             names of variables in the current problem
_conname{1.._ncons}             names of constraints in the current problem
_objname{1.._nobjs}             names of objectives in the current problem
```

Finally, the following synonyms for the components are made available:

```ampl
_var{1.._nvars}           synonyms for variables in the current problem
_con{1.._ncons}           synonyms for constraints in the current problem
_obj{1.._nobjs}           synonyms for objectives in the current problem
```

These synonyms `let` you refer to components by number, rather than by the usual indexed
names. Using the variables as an example, \_var[5] refers to the fifth variable in the
problem, \_var[5].ub to its upper bound, \_var[5].rc to its reduced cost, and so
forth, while \_varname[5] is a string giving the variable's true AMPL name. Table A-13
lists all of the generic synonyms for sets, variables, and the like.

Generic names are useful for tabulating properties of all variables, where the variables
have been defined in several different `var` declarations:

```ampl
ampl: model net3.mod
ampl: data net3.dat
ampl: solve;
MINOS 5.5: optimal solution found.

3 iterations, objective 1819
ampl: display {j in 1.._nvars}
ampl?   (_varname[j],_var[j],_var[j].ub,_var[j].rc);
:            _varname[j]      _var[j] _var[j].ub   _var[j].rc :=
1       "PD_Ship['NE']"          250      250     -0.5
2       "PD_Ship['SE']"          200      250     -1.11022e-16
3       "DW_Ship['NE','BOS']"     90       90      0
4       "DW_Ship['NE','EWR']"    100      100     -1.1
5       "DW_Ship['NE','BWI']"     60      100      0
6       "DW_Ship['SE','EWR']"     20      100      2.22045e-16
7       "DW_Ship['SE','BWI']"     60      100      2.22045e-16
8       "DW_Ship['SE','ATL']"     70       70      0
9       "DW_Ship['SE','MCO']"     50       50      0
;
```

Another use is to list all variables having some property, such as being away from the
upper bound in the optimal solution:

```ampl
ampl: display {j in 1.._nvars:
ampl?    _var[j] < _var[j].ub - 0.00001} _varname[j];
_varname[j] [*] :=
2 "PD_Ship['SE']"
5 "DW_Ship['NE','BWI']"
6 "DW_Ship['SE','EWR']"
7 "DW_Ship['SE','BWI']"
;
```

The same comments apply to constraints and objectives. More precise formatting of this
information can be obtained with printf (12.4, A.16) instead of `display`.

As in the case of the expand command, these parameters and generic synonyms
reflect the modeler's view of the problem; their values are determined from the `model`
and `data` as they were initially read and translated. AMPL's presolve phase may make significant
simplifications to the problem before it is sent to the solver. To work with
parameters and generic synonyms that reflect the solver's view of the problem following
presolve, replace _ by \_s in the names given above; for example in the case of variables,
use \_snvars, \_svarname and \_svar.

Additional predefined sets and parameters represent the names and dimensions (arities)
of the `model` components. They are summarized in A.19.4.

**Resource listings**

Changing option show_stats from its default of 0 to a nonzero value requests summary
statistics on the size of the optimization problem that AMPL generates:

```ampl
ampl:   model steelT.mod;
ampl:   data steelT.dat;
ampl:   option show_stats 1;
ampl:   solve;
Presolve eliminates 2 constraints and 2 variables.
Adjusted problem:
24 variables, all linear
12 constraints, all linear; 38 nonzeros
1 linear objective; 24 nonzeros.

MINOS 5.5: optimal solution found.

15 iterations, objective 515033
```

Additional lines report the numbers of `integer` and variables and nonlinear components
where appropriate.

Changing option times from its default of 0 to a nonzero value requests a summary
of the AMPL translator's time and memory requirements. Similarly, by changing option
gentimes to a nonzero value, you can get a detailed summary of the resources that
AMPL's genmod phase consumes in generating a `model` instance.
When AMPL appears to hang or takes much more time than expected, the `display` produced
by gentimes can help associate the difficulty with a particular `model` component.
Typically, some parameter, variable or constraint has been indexed over a set far
larger than intended or anticipated, with the result that excessive amounts of time and
memory are required.

The timings given by these commands apply only to the AMPL translator, not to the
solver. A variety of predefined parameters (Table A-14) `let` you work with both AMPL
and solver times. For example, _solve_time always equals total CPU seconds
required for the most recent `solve` command, and _ampl_time equals total CPU seconds
for AMPL excluding time spent in solvers and other external programs.

Many solvers also have directives for requesting breakdowns of the `solve` time. The
specifics vary considerably, however, so information on requesting and interpreting these
timings is provided in the documentation of AMPL's links to individual solvers, rather
than in this book.